# 01 Transfer (queue item 1, stage 1)

Conditions: **solo** (no routing), **transfer_unverified** (the better of two agents sends its artifact; the receiver adopts blindly), **transfer_verified** (the receiver adopts only if the payload beats its incumbent on 100 seeded self-play games). Two agents, 10 generations after the warm start, 3 population seeds, paired deals. The student is the agent with the lower generation-0 score.

**Scope: stub backend.** Agents are perturbed copies of anchor rule lists and there is no sabotage in this experiment, so verification can only cost (false rejections) here; this run validates the experiment pipeline, pairing and statistics, not the hypothesis.

In [1]:
import json, os, sys
sys.path.insert(0, 'src')
%matplotlib inline
import numpy as np
from culture.analysis.transfer import load_experiment, pairing_checks, curves, paired_deltas
from culture.analysis import figures
exp = load_experiment('runs/stage1')
pairing_checks(exp)

{'warm_starts_identical': True,
 'student_identical': True,
 'final_deals_identical': True,
 'conditions': ['solo', 'transfer_unverified', 'transfer_verified'],
 'population_seeds': [0, 1, 2]}

In [2]:
c = curves(exp)
figures.conditions_vs_tokens({k: v for k, v in c.items()}, 'Student score vs cumulative population tokens (IQM over 3 population seeds, bootstrap band)', 'student self-play score');

In [3]:
for k, v in c.items():
    print(f"{k:22s} final IQM {v['iqm'][-1]:5.2f}  per seed {np.round(v['per_seed'][:, -1], 2)}  tokens {v['tokens'][-1]:,.0f}")

solo                   final IQM  6.91  per seed [ 4.95  0.24 15.53]  tokens 169,297
transfer_unverified    final IQM 16.19  per seed [15.29 15.2  18.09]  tokens 196,584
transfer_verified      final IQM 13.47  per seed [14.98  7.74 17.68]  tokens 189,813


## Paired deltas

Game-level differences of the two conditions' final student artifacts on the identical final-generation deals (100 games per population seed), bootstrap stratified by population seed. With 3 population seeds the interval is conditional on these three populations; the per-seed means show the between-population spread, which is the honest uncertainty at this n.

In [4]:
rows = paired_deltas(exp, [('transfer_verified', 'solo'), ('transfer_unverified', 'solo'), ('transfer_verified', 'transfer_unverified')])
figures.paired_deltas(rows, 'Final student score, paired differences (95% stratified bootstrap)');
for r in rows:
    print(r['label'], round(r['mean'], 2), (round(r['lo'], 2), round(r['hi'], 2)), 'per seed', np.round(r['per_seed_mean'], 2), 'AUC delta per seed', np.round(r['auc_delta_per_seed'], 2), 'token ratio', round(r['tokens_ratio'], 2))

transfer_verified - solo 6.56 (6.17, 6.96) per seed [10.03  7.5   2.15] AUC delta per seed [9.31 7.09 0.64] token ratio 1.12
transfer_unverified - solo 9.29 (8.91, 9.68) per seed [10.34 14.96  2.56] AUC delta per seed [ 9.44 10.04  1.17] token ratio 1.16
transfer_verified - transfer_unverified -2.73 (-2.96, -2.48) per seed [-0.31 -7.46 -0.41] AUC delta per seed [-0.13 -2.95 -0.53] token ratio 0.97


## Learning smoke test (spec section 12)

Run through the real generation loop with LLM revision off; the same checks are asserted in `tests/test_smoke_learning.py`.

In [5]:
import subprocess
print(subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_smoke_learning.py'], capture_output=True, text=True).stdout[-400:])

...                                                                      [100%]
3 passed in 9.55s

